# CS604 Colab launcher — Van

Runtime: **A100 40GB GPU**, High-RAM off. Images stay on Colab disk (`/content/data`). Checkpoints and logs go to the shared Drive folder **CS604_RUNS**.

The GitHub repo is private, so the clone cell will ask for a Personal Access Token (hidden). Do not paste the token into the notebook text or commit it.

**Drive:** everyone must have `CS604_RUNS` under *My Drive* (create it there, or open the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR) → Organize → **Add shortcut to Drive** → My Drive, name it `CS604_RUNS`). A Drive *link* is not a file path Colab can write to.

A copy of `colab_launcher.ipynb` set up for Van's full fine-tune (`configs/van_full_ft_proto_llrd.yaml`, branch `van-dev`). Differences:

- `BRANCH`, `METHOD` and `TEAM` are already set.
- Prediction finds the run folder that training just created and writes `val_team_{TEAM}.txt` and `.zip` inside it (not in the shared `CS604_RUNS` root). It skips the test split until test data is released.
- With `AUTO_DISCONNECT = True`, the runtime ends after the last cell, or as soon as training fails, so an idle GPU does not keep using compute units. Drive is flushed first.

Use **Runtime → Run all**.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "van-dev"
METHOD = "van_full_ft_proto_llrd"  # configs/<METHOD>.yaml
TEAM = "08"
PROTO_PT = f"{DRIVE_OUT}/prototypes.pt"
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 4b. Build class prototypes (once per backbone)

Van's ~43.9% zero-shot head. Run **once**, then reuse `PROTO_PT` for `prototype_probe` / `prototype_lora_r8`. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/prototype_probe.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## Review settings (table)

Loads `configs/{METHOD}.yaml` (merged with `base.yaml`) so you can check every parameter before spending a GPU hour.

In [ ]:
import pandas as pd

from src.utils import apply_overrides, load_config

methods = ["full_ft", "linear_probe", "lora_r8", "bitfit", "last_blocks", "prototype_probe", "prototype_lora_r8", "adapter", "adaptformer"]
merged = {name: load_config(f"configs/{name}.yaml").to_dict() for name in methods}
keys = []
for d in merged.values():
    for k in d:
        if k not in keys:
            keys.append(k)

compare = pd.DataFrame(
    {name: [merged[name].get(k, "") for k in keys] for name in methods},
    index=keys,
)
compare.index.name = "parameter"
print("Resolved parameters per method (base.yaml already merged in):")
display(compare)

chosen = load_config(f"configs/{METHOD}.yaml")
apply_overrides(chosen, data_root=DATA_DIR, output_dir=DRIVE_OUT)
resolved = pd.DataFrame(list(chosen.to_dict().items()), columns=["parameter", "value"])
print(f"\nThis run (METHOD={METHOD!r}, plus Colab data_root / output_dir):")
display(resolved)

## 5. Train (checkpoints on Drive)

In [ ]:
import subprocess
import sys
import time

TRAIN_STARTED = time.time()
proc = subprocess.Popen(
    [sys.executable, "-u", "-m", "src.train", "--config", f"configs/{METHOD}.yaml",
     "--data_root", DATA_DIR, "--output_dir", DRIVE_OUT],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
if proc.wait() != 0:
    end_runtime(f"training failed (exit {proc.returncode}); see the output above")
    raise SystemExit("training failed")

## 6. Predict

Uses the newest run folder created since training started; set `RUN_NAME` by hand only to predict from an older run. Uses `last.pt`: this config trains on all images, so there is no labelled validation and no `best.pt`. Writes `val_team_{TEAM}.txt` and `.zip` into the run folder.

In [ ]:
import shutil
from pathlib import Path

RUN_NAME = None  # None: the run folder training just created
if RUN_NAME is None:
    started = globals().get("TRAIN_STARTED", 0)
    runs = [p for p in Path(DRIVE_OUT).iterdir() if p.is_dir() and p.stat().st_mtime >= started
            and (p / "config.yaml").is_file()]
    assert runs, "No run folder from this session; set RUN_NAME by hand"
    RUN_NAME = max(runs, key=lambda p: p.stat().st_mtime).name
RUN_DIR = Path(DRIVE_OUT) / RUN_NAME
CKPT = RUN_DIR / "last.pt"
assert CKPT.is_file(), f"Checkpoint missing: {CKPT}"
print("Predicting from", CKPT)

!python -m src.predict --config {RUN_DIR}/config.yaml --ckpt {CKPT} --split val --team {TEAM} --data_root {DATA_DIR} --out_dir {RUN_DIR}
shutil.make_archive(str(RUN_DIR / f"val_team_{TEAM}"), "zip", root_dir=RUN_DIR, base_dir=f"val_team_{TEAM}.txt")
print("Submit:", RUN_DIR / f"val_team_{TEAM}.zip")
if (Path(DATA_DIR) / "test.txt").is_file() and (Path(DATA_DIR) / "test").is_dir():
    !python -m src.predict --config {RUN_DIR}/config.yaml --ckpt {CKPT} --split test --team {TEAM} --data_root {DATA_DIR} --out_dir {RUN_DIR}
else:
    print("Test data not available; skipping test prediction.")

## 7. End the runtime

Flushes Drive and releases the GPU when `AUTO_DISCONNECT = True`.

In [ ]:
end_runtime("all cells finished; ending the runtime")